# buffer-copy_-inplace — worked example 3: copy_ keeps destination dtype when source dtype differs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `buffer-copy_-inplace`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`tensor.copy_(src)` always keeps the *destination's* dtype: the source is cast to the destination dtype as it is written. So copying a float64 source into a float32 buffer leaves a float32 buffer (with rounded values), not a float64 one. The buffer's identity and dtype are both preserved — only the numbers change.

## Worked solution

**Goal.** Copy a float64 EMA update into a float32 buffer and confirm the buffer stays float32 with the same identity.

**Step 1 — note the destination dtype.** The buffer is `float32`. We deliberately build the source in `float64` to show coercion.

**Step 2 — compute the source.** `src = (running_mean.double() * (1 - momentum)) + batch_mean.double() * momentum` produces a `float64` tensor. (We cast to double to make the dtype mismatch explicit.)

**Step 3 — copy in place.** `running_mean.copy_(src)` casts each `float64` element down to `float32` as it writes into the buffer's storage. The buffer remains `float32` and keeps its `id`. PyTorch's `copy_` never changes the destination dtype — it coerces the source to match. That guarantees a registered float32 buffer never silently becomes float64.

In [ ]:
def copy_with_coercion(running_mean: Tensor, batch_mean: Tensor, momentum: float) -> None:
    src = running_mean.double() * (1 - momentum) + batch_mean.double() * momentum
    running_mean.copy_(src)

t.manual_seed(0)
running_mean = t.zeros(3, dtype=t.float32)
batch_mean = t.tensor([1.0, 2.0, 3.0], dtype=t.float32)
before = id(running_mean)
copy_with_coercion(running_mean, batch_mean, momentum=0.25)
print("dtype still float32:", running_mean.dtype == t.float32)
print("id preserved:", id(running_mean) == before)
print("running_mean:", running_mean)